# Road Accidents Comparison: United States vs United Kingdom

## Introduction
This notebook compares road accident patterns, severity, temporal trends, and contributing factors between the US and UK.

**Objectives:**
- Analyze temporal patterns (year, month, hour, day of week)
- Compare severity distributions
- Explore weather, road conditions, and location factors
- Identify similarities and key differences
- Provide data-driven insights

**Datasets:**
- US: 500,000 sample of US Accidents (2016–2023)
- UK: Official UK Road Casualty Statistics (Last 5 years)

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import os
from urllib.request import urlretrieve
import zipfile
import gdown #requires pip install gdown or conda install conda-forge::gdown


# import geopandas as gpd
# import folium

%matplotlib inline


In [ ]:

# ================== DIRECT LOAD FROM GOV.UK ==================

uk_collision_url = "https://data.dft.gov.uk/road-accidents-safety-data/dft-road-casualty-statistics-collision-last-5-years.csv"

print("Downloading UK Collisions data...")
uk_acc = pd.read_csv(uk_collision_url, low_memory=False)
years = uk_acc['collision_year'].unique()
years.sort()

print("UK Collisions Shape:", uk_acc.shape)
print("Columns:", uk_acc.columns.tolist())
print("Years covered:", years)

In [ ]:
# File ID from the official sampled dataset
file_id = "1U3u8QYzLjnEaSurtZfSAS_oh9AT2Mn8X"
url = f"https://drive.google.com/uc?id={file_id}"

output_zip = "US_Accidents_500k.zip"

print("Downloading US Accidents 500k sample using gdown...")
gdown.download(url, output_zip, quiet=False)

print("Download finished. Extracting files...")

In [ ]:
import zipfile

# Extract the zip file
with zipfile.ZipFile(output_zip, 'r') as zip_ref:
    zip_ref.extractall(".")
    print("Extracted files:")
    print(os.listdir("."))

# Find the CSV file
csv_files = [f for f in os.listdir(".") if f.endswith('.csv')]
print("CSV files found:", csv_files)

# Load the main CSV
if csv_files:
    csv_name = csv_files[0]
    us_acc = pd.read_csv(csv_name, low_memory=False)
    print(f"✅ Successfully loaded {csv_name}")
    print("Shape:", us_acc.shape)
    print("Years:", sorted(us_acc['Start_Time'].str[:4].unique()))
else:
    print("No CSV found!")

In [48]:
def preprocess_us(df):
    df = df.copy()
    df['Start_Time'] = pd.to_datetime(df['Start_Time'], errors='coerce')
    df['Year'] = df['Start_Time'].dt.year.astype('Int64')
    df['Month'] = df['Start_Time'].dt.month
    df['Hour'] = df['Start_Time'].dt.hour
    df['Day_of_Week'] = df['Start_Time'].dt.day_name()
    return df

def preprocess_uk(df):
    df = df.copy()
    
    df.rename(columns={
        'collision_year': 'Year',
        'collision_severity': 'Severity',           # 1=Fatal, 2=Serious, 3=Slight
        'date': 'Date',
        'time': 'Time',
        'latitude': 'Latitude',
        'longitude': 'Longitude',
        'day_of_week': 'Day_of_Week',
        'speed_limit': 'Speed_Limit',
        'urban_or_rural_area': 'Urban_Rural'
    }, inplace=True)
    
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df['Hour'] = pd.to_datetime(df['Time'], format='%H:%M', errors='coerce').dt.hour
    
    severity_map = {1: 'Fatal', 2: 'Serious', 3: 'Slight'}
    df['Severity_Label'] = df['Severity'].map(severity_map)
    
    return df

uk_clean = preprocess_uk(uk_acc)
us_clean = preprocess_us(us_acc)
print(uk_clean.head())

/tmp/ipykernel_121130/3233697762.py:25: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Date'] = pd.to_datetime(df['Date'], errors='coerce')


  collision_index  Year collision_ref_no  location_easting_osgr  \
0   2021170H10421  2021        170H10421               447098.0   
1   2021170H11231  2021        170H11231               450486.0   
2   2020170M11750  2020        170M11750               449694.0   
3   2021170M31761  2021        170M31761               449744.0   
4   2021170S10441  2021        170S10441               445971.0   

   location_northing_osgr  Longitude   Latitude  police_force  Severity  \
0                532997.0  -1.270905  54.689833            17         3   
1                533118.0  -1.218333  54.690592            17         3   
2                519733.0  -1.232884  54.570397            17         3   
3                514217.0  -1.233040  54.520825            17         3   
4                520834.0  -1.290292  54.580641            17         3   

   number_of_vehicles  ...  Urban_Rural  \
0                   2  ...            2   
1                   2  ...            1   
2                

In [50]:
uk_clean.head(5)

,collision_index,Year,collision_ref_no,location_easting_osgr,location_northing_osgr,Longitude,Latitude,police_force,Severity,number_of_vehicles,...,Urban_Rural,did_police_officer_attend_scene_of_accident,trunk_road_flag,lsoa_of_accident_location,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight,Hour,Severity_Label
0,2021170H10421,2021,170H10421,447098.0,532997.0,-1.270905,54.689833,17,3,2,...,2,1,2,E01011959,-1,0,0.293588,0.706412,22,Slight
1,2021170H11231,2021,170H11231,450486.0,533118.0,-1.218333,54.690592,17,3,2,...,1,2,2,E01011973,-1,0,0.017448,0.982552,15,Slight
2,2020170M11750,2020,170M11750,449694.0,519733.0,-1.232884,54.570397,17,3,2,...,1,1,2,E01012092,-1,0,0.128730,0.871270,18,Slight
3,2021170M31761,2021,170M31761,449744.0,514217.0,-1.233040,54.520825,17,3,1,...,2,1,2,E01032553,-1,0,0.182698,0.817302,16,Slight
4,2021170S10441,2021,170S10441,445971.0,520834.0,-1.290292,54.580641,17,3,3,...,2,1,1,E01012258,-1,0,0.016094,0.983906,9,Slight


In [51]:
us_clean.head(5)

,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight,Year,Month,Hour,Day_of_Week
0,A-2047758,Source2,2,2019-06-12 10:10:56,2019-06-12 10:55:58,30.641211,-91.153481,NaN,NaN,0.000,...,True,False,Day,Day,Day,Day,2019,6.0,10.0,Wednesday
1,A-4694324,Source1,2,NaT,2022-12-04 01:56:53.000000000,38.990562,-77.399070,38.990037,-77.398282,0.056,...,False,False,Night,Night,Night,Night,<NA>,NaN,NaN,NaN
2,A-5006183,Source1,2,NaT,2022-08-20 15:22:45.000000000,34.661189,-120.492822,34.661189,-120.492442,0.022,...,True,False,Day,Day,Day,Day,<NA>,NaN,NaN,NaN
3,A-4237356,Source1,2,2022-02-21 17:43:04,2022-02-21 19:43:23,43.680592,-92.993317,43.680574,-92.972223,1.054,...,False,False,Day,Day,Day,Day,2022,2.0,17.0,Monday
4,A-6690583,Source1,2,2020-12-04 01:46:00,2020-12-04 04:13:09,35.395484,-118.985176,35.395476,-118.985995,0.046,...,False,False,Night,Night,Night,Night,2020,12.0,1.0,Friday
